# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploration, and preliminary processing of the FAIR^2 dataset, which details protein detection in human extracellular vesicles, using the `mlcroissant` library.

## Dataset Source
The dataset source is a [Croissant schema](https://mlcommons.org/croissant/) accessible via a public schema URL.

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect the dataset object using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and print key dataset attributes
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s as parsed from the Croissant metadata.

In [ ]:
# List available record sets with @id
record_sets = dataset.metadata.record_sets

print(f"Total record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    # List fields
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis.

**Note:** All references are made using `@id` fields as best practice. If there is only one main/primary record set, use its `@id` in the code below.

In [ ]:
# Prepare to extract all record sets
import pprint
dataframes = {}

for record_set in record_sets:
    recset_id = record_set.id
    print(f"Loading records from RecordSet: {recset_id} ({record_set.name})")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample Records (first 2):")
    pprint.pprint(df.head(2).to_dict(orient='records'))

# For further steps, select the main record set id (typically contains protein-level data)
# For this dataset, use the first one detected for demonstration:
main_record_set_id = record_sets[0].id if record_sets else None
assert main_record_set_id is not None, "No record sets found."

print(f"\nMain RecordSet for analysis: {main_record_set_id}")
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps. This example shows filtering a numeric field, normalizing, and grouping. **Replace placeholder field `@id`s with those applicable to this dataset.**

In [ ]:
# For demonstration, pick a numeric field (e.g., 'cr:field/protein_mw' for molecular weight or similar)
# List columns to help select fields:
df = dataframes[main_record_set_id]
print("Available columns (Croissant field @id):")
for col in df.columns:
    print(f"- {col}")

# Based on commonly used columns in protein datasets, such as:
# 'cr:field/mw' - molecular weight
# 'cr:field/abundance' - protein abundance
# For demonstration, choose the first numeric column available
import numpy as np
numeric_field = None
for c in df.columns:
    if np.issubdtype(df[c].dtype, np.number):
        numeric_field = c
        break
if numeric_field is None:
    # Try to convert fields likely to be numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c], errors='coerce')
            if np.issubdtype(df[c].dtype, np.number):
                numeric_field = c
                break
        except Exception:
            pass
assert numeric_field is not None, "No numeric field found for EDA."

threshold = df[numeric_field].quantile(0.90)  # show high-abundance/protein MW in top 10%
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (90th percentile):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by an appropriate field, e.g. sample or category
# Use any string/categorical-type column as group (other than the numeric)
group_field = None
for c in df.columns:
    if c != numeric_field and df[c].dtype == object:
        group_field = c
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped means by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the numeric field's distribution (histogram), and a barplot for grouped means, if groups are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field (original and normalized)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, ax=axes[0], color='skyblue')
axes[0].set_title(f"Distribution of {numeric_field}")

sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, ax=axes[1], color='salmon')
axes[1].set_title(f"Normalized {numeric_field} (top 10%)")
plt.tight_layout()
plt.show()

# Barplot for groups if available
if group_field is not None:
    plt.figure(figsize=(10, 5))
    # Top 10 groups by mean value
    sorted_means = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    sns.barplot(data=sorted_means.reset_index(), x=group_field, y=numeric_field, palette='Blues_d')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- Successfully loaded and summarized the FAIR^2 mass spectrometry dataset via Croissant schema.
- Inspected available record sets, examined field metadata, and previewed data content.
- Demonstrated exploratory filtering, normalization, and grouping by record set field `@id`.
- Visualized distribution and grouped summaries for a selected numeric field.

See [FAIR^2 dataset page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for dataset documentation and further information.